
# 🌱 Crop Recommendation System (Machine Learning)

Recommends the best crop to grow based on **soil nutrients** (N, P, K, pH) and
**weather conditions** (temperature, humidity, rainfall).

This notebook:
1. Builds/loads a dataset of soil & weather readings labeled by crop
2. Trains a Random Forest classifier
3. Evaluates accuracy
4. Lets you predict the best crop for new readings

> Runs top-to-bottom in Google Colab. No external files required — a
> realistic synthetic dataset is generated automatically. If you have your
> own CSV (e.g. the popular Kaggle "Crop Recommendation Dataset" with columns
> `N, P, K, temperature, humidity, ph, rainfall, label`), see **Step 2b** to
> use it instead.


## Step 1 — Install & import libraries

In [ ]:

!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import joblib

np.random.seed(42)



## Step 2a — Generate a synthetic training dataset

Each crop has a realistic "ideal range" for every parameter (based on typical
agronomic guidelines). We sample many random points inside (and slightly
around) each crop's ideal range to build a labeled dataset the model can
learn from.


In [ ]:

# Ideal (min, max) ranges per crop — used to synthesize realistic samples
CROP_RANGES = {
    "rice":       {"N": (80, 120), "P": (40, 60),  "K": (40, 60),  "ph": (5.5, 7.0), "temperature": (20, 35), "humidity": (70, 90), "rainfall": (150, 300)},
    "wheat":      {"N": (60, 100), "P": (30, 50),  "K": (30, 50),  "ph": (6.0, 7.5), "temperature": (10, 25), "humidity": (40, 60), "rainfall": (50, 100)},
    "maize":      {"N": (80, 120), "P": (40, 60),  "K": (40, 60),  "ph": (5.5, 7.5), "temperature": (18, 32), "humidity": (50, 70), "rainfall": (60, 120)},
    "cotton":     {"N": (100,140), "P": (30, 50),  "K": (30, 50),  "ph": (5.5, 8.0), "temperature": (21, 35), "humidity": (40, 70), "rainfall": (60, 110)},
    "sugarcane":  {"N": (100,150), "P": (40, 70),  "K": (60, 100), "ph": (6.0, 7.5), "temperature": (20, 35), "humidity": (60, 85), "rainfall": (100, 200)},
    "chickpea":   {"N": (20, 40),  "P": (40, 60),  "K": (20, 40),  "ph": (6.0, 8.0), "temperature": (15, 28), "humidity": (30, 60), "rainfall": (30, 65)},
    "potato":     {"N": (80, 120), "P": (50, 70),  "K": (80, 120), "ph": (5.0, 6.5), "temperature": (15, 24), "humidity": (60, 80), "rainfall": (40, 90)},
    "tomato":     {"N": (60, 100), "P": (40, 60),  "K": (60, 100), "ph": (6.0, 6.8), "temperature": (18, 28), "humidity": (50, 70), "rainfall": (40, 80)},
    "mango":      {"N": (30, 60),  "P": (20, 40),  "K": (40, 60),  "ph": (5.5, 7.5), "temperature": (24, 35), "humidity": (40, 60), "rainfall": (75, 150)},
    "banana":     {"N": (100,150), "P": (30, 50),  "K": (150,200), "ph": (5.5, 7.0), "temperature": (22, 32), "humidity": (70, 90), "rainfall": (120, 220)},
    "coffee":     {"N": (60, 100), "P": (20, 40),  "K": (40, 60),  "ph": (6.0, 6.8), "temperature": (18, 28), "humidity": (60, 80), "rainfall": (150, 250)},
    "jute":       {"N": (60, 100), "P": (30, 50),  "K": (30, 50),  "ph": (6.0, 7.5), "temperature": (24, 35), "humidity": (70, 90), "rainfall": (150, 250)},
}

FEATURES = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
SAMPLES_PER_CROP = 200  # increase for a larger dataset

def synthesize_dataset(ranges, samples_per_crop=200, spread=0.15, seed=42):
    rng = np.random.default_rng(seed)
    rows = []
    for crop, bounds in ranges.items():
        for _ in range(samples_per_crop):
            row = {}
            for feat in FEATURES:
                low, high = bounds[feat]
                width = high - low
                # allow a little spillover outside the ideal range for realism
                low_ext, high_ext = low - spread * width, high + spread * width
                row[feat] = rng.uniform(low_ext, high_ext)
            row["label"] = crop
            rows.append(row)
    return pd.DataFrame(rows)

df = synthesize_dataset(CROP_RANGES, SAMPLES_PER_CROP)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle
print(f"Dataset shape: {df.shape}")
df.head()



## Step 2b — (Optional) Use your own CSV instead

If you have a real dataset (e.g. Kaggle's Crop Recommendation Dataset with
columns `N, P, K, temperature, humidity, ph, rainfall, label`), upload it and
uncomment the code below to use it instead of the synthetic data.


In [ ]:

# from google.colab import files
# uploaded = files.upload()          # choose your .csv file
# csv_name = list(uploaded.keys())[0]
# df = pd.read_csv(csv_name)
# print(df.shape)
# df.head()


## Step 3 — Explore the dataset

In [ ]:

print("Crops in dataset:", df['label'].nunique())
print(df['label'].value_counts())

df.describe()


In [ ]:

plt.figure(figsize=(10, 6))
sns.heatmap(df[FEATURES].corr(), annot=True, cmap="YlGnBu", fmt=".2f")
plt.title("Feature correlation")
plt.show()


## Step 4 — Prepare data & train the model

In [ ]:

X = df[FEATURES]
y = df["label"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)
print("Model trained.")


## Step 5 — Evaluate the model

In [ ]:

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc*100:.2f}%\n")

print(classification_report(y_test, y_pred, target_names=le.classes_))


In [ ]:

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.xticks(rotation=45, ha="right")
plt.show()


In [ ]:

importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
plt.figure(figsize=(8, 5))
sns.barplot(x=importances.values, y=importances.index, palette="viridis")
plt.title("Feature importance")
plt.xlabel("Importance")
plt.show()
importances



## Step 6 — Recommend a crop for new soil & weather readings

Change the values below to your own soil test / weather readings, then run
the cell. The model returns the best crop plus the top-3 most likely crops
with probabilities.


In [ ]:

def recommend_crop(N, P, K, temperature, humidity, ph, rainfall, top_n=3):
    sample = pd.DataFrame([{
        "N": N, "P": P, "K": K,
        "temperature": temperature, "humidity": humidity,
        "ph": ph, "rainfall": rainfall,
    }])[FEATURES]

    probs = model.predict_proba(sample)[0]
    top_idx = np.argsort(probs)[::-1][:top_n]

    print("Input conditions:")
    for k, v in sample.iloc[0].items():
        print(f"  {k:>12}: {v}")

    print(f"\nTop {top_n} recommended crops:")
    for rank, idx in enumerate(top_idx, start=1):
        crop = le.classes_[idx]
        print(f"  {rank}. {crop.title():<10} {probs[idx]*100:5.1f}%")

    best = le.classes_[top_idx[0]]
    print(f"\n>>> Best match: {best.upper()}")
    return best

# Example: try your own readings here
recommend_crop(
    N=90, P=42, K=43,
    temperature=26, humidity=82,
    ph=6.5, rainfall=210,
)


## Step 7 — Save the model (optional)

In [ ]:

joblib.dump(model, "crop_recommendation_model.pkl")
joblib.dump(le, "crop_label_encoder.pkl")
print("Saved model and label encoder.")

# Uncomment to download the files to your computer
# from google.colab import files
# files.download("crop_recommendation_model.pkl")
# files.download("crop_label_encoder.pkl")
